# Unique cell-type marker enrichment for two AEP DESeq2 comparisons

This notebook tests whether genes uniquely assigned to each fine-grained cell type in `unique_celltypes.table` are overrepresented among significantly upregulated or downregulated genes.

Unlike a within-cell-type expression quantile, these marker sets are intended to represent cell-type specificity. Results describe enrichment of cell-type marker genes, not changes in cell abundance.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

## Configuration

The primary analysis uses the 32 fine-grained cell-type columns and excludes the broader `C_*` class and `L_*` lineage columns. Tests with fewer than `MIN_MAPPED_MARKERS` mapped markers are retained in the all-tests table but are not called significant.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

UNIQUE_CELLTYPE_FILE = PROJECT_ROOT / "data" / "unique_celltypes.table"
ECOKD_DESEQ_FILE = PROJECT_ROOT / "data" / "EcoKD1_Eco1KD_B8_vs_control_B8_results.csv"
RECOLONIZATION_DESEQ_FILE = PROJECT_ROOT / "data" / "HydraRecolonization_Wild_vs_GF_results.csv"
CONVENTIONALIZED_DESEQ_FILE = PROJECT_ROOT / "data" / "HydraRecolonization_Conventionalized_vs_GF_results.csv"
CURVIBACTER_DESEQ_FILE = PROJECT_ROOT / "data" / "HydraRecolonization_Cvbct_vs_GF_results.csv"
HOMOSERINE_DESEQ_FILE = PROJECT_ROOT / "data" / "HydraAHL_3OHC12_vs_control_results.csv"

OUTPUT_DIR = PROJECT_ROOT / "results" / "unique_marker_celltype_enrichment_deseq2_comparisons"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PADJ_CUTOFF = 0.05
MIN_ABS_LOG2_FOLD_CHANGE = 0.0
FDR_CUTOFF = 0.05
MIN_MAPPED_MARKERS = 10

## Read DESeq2 results and unique cell-type markers

Rows in `unique_celltypes.table` contain an extra leading numeric index. With pandas' parser, the AEP identifier is correctly exposed as the `ID` column. A marker is assigned to a cell type when that cell-type column contains a positive value.

The enrichment universe is all genes in each DESeq2 results table with a valid, unique gene suffix. Marker annotations are left-joined onto that universe; genes absent from `unique_celltypes.table` are retained and treated as having no unique cell-type assignment. This ensures the Fisher tests compare marker genes with all tested DESeq2 genes, assuming every gene in the DESeq2 table was eligible for marker assignment.


In [ ]:
def read_deseq_csv(path):
    table = pd.read_csv(path, index_col=0)
    table.index = table.index.astype(str)
    table.index.name = "aep_id"
    table = table.reset_index()
    for column in ["baseMean", "log2FoldChange", "lfcSE", "stat", "pvalue", "padj"]:
        if column in table.columns:
            table[column] = pd.to_numeric(table[column], errors="coerce")
    required = {"aep_id", "log2FoldChange", "padj"}
    missing = required.difference(table.columns)
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {sorted(missing)}")
    if not table["aep_id"].is_unique:
        raise ValueError(f"{path.name} contains duplicated AEP gene IDs.")
    return table


def gene_suffix(values):
    return values.astype(str).str.extract(r"(G\d+)$", expand=False)


unique_raw = pd.read_csv(UNIQUE_CELLTYPE_FILE, sep=";")
if "ID" not in unique_raw.columns:
    raise ValueError("unique_celltypes.table is missing the ID column.")

celltype_columns = [
    column for column in unique_raw.columns
    if column != "ID" and not column.startswith(("L_"))
]
unique_markers = unique_raw.loc[unique_raw["ID"].notna(), ["ID", *celltype_columns]].copy()
unique_markers = unique_markers.rename(columns={"ID": "marker_aep_id"})
unique_markers["gene_suffix"] = gene_suffix(unique_markers["marker_aep_id"])
unique_markers = unique_markers.loc[unique_markers["gene_suffix"].notna()].copy()
unique_markers[celltype_columns] = (
    unique_markers[celltype_columns].apply(pd.to_numeric, errors="coerce").fillna(0).gt(0).astype(int)
)
unique_markers = unique_markers.groupby("gene_suffix", as_index=False)[celltype_columns].max()

DESEQ_TABLES = {
    "EcoKD1_Eco1KD_B8_vs_control_B8": read_deseq_csv(ECOKD_DESEQ_FILE),
    "HydraRecolonization_Wild_vs_GF": read_deseq_csv(RECOLONIZATION_DESEQ_FILE),
    "HydraRecolonization_Conventionalized_vs_GF" : read_deseq_csv(CONVENTIONALIZED_DESEQ_FILE),
    "HydraRecolonization_Curvibacter_vs_GF" : read_deseq_csv(CURVIBACTER_DESEQ_FILE),
    "HydraAHL_3OHC12_vs_control" : read_deseq_csv(HOMOSERINE_DESEQ_FILE)
}

marker_counts = unique_markers[celltype_columns].sum().sort_values(ascending=False).rename("unique_markers")
print(f"Genes represented in unique-celltype table: {len(unique_markers):,}")
print(f"Fine-grained cell types: {len(celltype_columns)}")
display(marker_counts.to_frame())

In [ ]:
for i in marker_counts.keys():
    if i.endswith("_Head"):
        print(i)

## Run comparison-specific enrichment analyses

For each direction and cell type, a one-sided Fisher exact test asks whether unique markers are overrepresented among significantly upregulated or downregulated genes relative to all genes in that comparison's DESeq2 results table. Genes with no matching row in `unique_celltypes.table` remain in the background with no cell-type assignment. Benjamini-Hochberg correction is applied separately within each direction and comparison, as in the original notebook.


In [ ]:
def benjamini_hochberg(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues)
    ranked = pvalues[order]
    adjusted_ranked = np.minimum.accumulate(
        (ranked * len(ranked) / np.arange(1, len(ranked) + 1))[::-1]
    )[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted


def analyze_comparison(comparison, deseq):
    comparison_output_dir = OUTPUT_DIR / comparison
    comparison_output_dir.mkdir(parents=True, exist_ok=True)

    deseq_with_suffix = deseq.copy()
    deseq_with_suffix["gene_suffix"] = gene_suffix(deseq_with_suffix["aep_id"])
    if deseq_with_suffix["gene_suffix"].isna().any() or deseq_with_suffix["gene_suffix"].duplicated().any():
        raise ValueError(f"{comparison}: DESeq2 gene suffixes are missing or duplicated.")
    # Keep every gene tested by DESeq2 in the universe. Genes without a marker-table
    # match are retained and treated as unassigned to every cell type.
    mapped = deseq_with_suffix.merge(unique_markers, on="gene_suffix", how="left", validate="one_to_one")
    mapped[celltype_columns] = mapped[celltype_columns].fillna(0).astype(int)
    mapped["direction"] = "not_significant"
    significant = mapped["padj"].lt(PADJ_CUTOFF) & mapped["log2FoldChange"].abs().ge(MIN_ABS_LOG2_FOLD_CHANGE)
    mapped.loc[significant & mapped["log2FoldChange"].gt(0), "direction"] = "upregulated"
    mapped.loc[significant & mapped["log2FoldChange"].lt(0), "direction"] = "downregulated"

    marker_coverage = pd.DataFrame({
        "comparison": comparison,
        "celltype": celltype_columns,
        "markers_in_unique_table": unique_markers[celltype_columns].sum().values,
        "markers_in_deseq2_background": mapped[celltype_columns].sum().values,
    })
    marker_coverage["tested_marker_fraction"] = (
        marker_coverage["markers_in_deseq2_background"] / marker_coverage["markers_in_unique_table"]
    )

    overview = pd.DataFrame([
        {"comparison": comparison, "metric": "DESeq2 genes", "count": len(deseq)},
        {"comparison": comparison, "metric": "genes in DESeq2 enrichment background", "count": len(mapped)},
        {"comparison": comparison, "metric": "background genes with at least one unique cell-type assignment", "count": int(mapped[celltype_columns].any(axis=1).sum())},
        {"comparison": comparison, "metric": "significant upregulated mapped genes", "count": int(mapped["direction"].eq("upregulated").sum())},
        {"comparison": comparison, "metric": "significant downregulated mapped genes", "count": int(mapped["direction"].eq("downregulated").sum())},
    ])

    def enrichment_for_direction(direction):
        selected = mapped["direction"].eq(direction)
        rows = []
        for celltype in celltype_columns:
            annotated = mapped[celltype].eq(1)
            a = int((selected & annotated).sum())
            b = int((selected & ~annotated).sum())
            c = int((~selected & annotated).sum())
            d = int((~selected & ~annotated).sum())
            odds_ratio, pvalue = fisher_exact([[a, b], [c, d]], alternative="greater")
            rows.append({
                "comparison": comparison,
                "direction": direction,
                "celltype": celltype,
                "selected_markers": a,
                "selected_total": a + b,
                "background_markers": c,
                "background_total": c + d,
                "background_marker_total": a + c,
                "selected_fraction": a / (a + b) if a + b else np.nan,
                "background_fraction": c / (c + d) if c + d else np.nan,
                "odds_ratio": odds_ratio,
                "pvalue": pvalue,
            })
        result = pd.DataFrame(rows)
        result["fdr"] = benjamini_hochberg(result["pvalue"])
        result["passes_marker_count"] = result["background_marker_total"].ge(MIN_MAPPED_MARKERS)
        result["significantly_overrepresented"] = (
            result["passes_marker_count"] & result["fdr"].lt(FDR_CUTOFF) & result["odds_ratio"].gt(1)
        )
        return result.sort_values(["fdr", "pvalue", "odds_ratio"], ascending=[True, True, False])

    enrichment = pd.concat(
        [enrichment_for_direction("upregulated"), enrichment_for_direction("downregulated")],
        ignore_index=True,
    )
    significant_enrichment = enrichment.loc[enrichment["significantly_overrepresented"]].copy()

    contributors = []
    for row in significant_enrichment.itertuples(index=False):
        genes = mapped.loc[
            mapped["direction"].eq(row.direction) & mapped[row.celltype].eq(1),
            ["aep_id", "baseMean", "log2FoldChange", "pvalue", "padj"],
        ].copy()
        genes.insert(0, "celltype", row.celltype)
        genes.insert(0, "direction", row.direction)
        genes.insert(0, "comparison", comparison)
        contributors.append(genes)
    contributing_genes = pd.concat(contributors, ignore_index=True) if contributors else pd.DataFrame(
        columns=["comparison", "direction", "celltype", "aep_id", "baseMean", "log2FoldChange", "pvalue", "padj"]
    )

    mapped.insert(0, "comparison", comparison)
    mapped.to_csv(comparison_output_dir / "deseq_genes_with_unique_celltype_markers.csv", index=False)
    marker_coverage.to_csv(comparison_output_dir / "unique_marker_coverage.csv", index=False)
    overview.to_csv(comparison_output_dir / "pipeline_overview.csv", index=False)
    enrichment.to_csv(comparison_output_dir / "unique_marker_enrichment_all_tests.csv", index=False)
    significant_enrichment.to_csv(comparison_output_dir / "unique_marker_enrichment_significant.csv", index=False)
    contributing_genes.to_csv(comparison_output_dir / "genes_contributing_to_significant_enrichment.csv", index=False)

    return {"mapped": mapped, "coverage": marker_coverage, "overview": overview, "enrichment": enrichment, "significant": significant_enrichment, "contributors": contributing_genes}


comparison_results = {comparison: analyze_comparison(comparison, deseq) for comparison, deseq in DESEQ_TABLES.items()}

In [ ]:
combined_overview = pd.concat([result["overview"] for result in comparison_results.values()], ignore_index=True)
combined_coverage = pd.concat([result["coverage"] for result in comparison_results.values()], ignore_index=True)
combined_enrichment = pd.concat([result["enrichment"] for result in comparison_results.values()], ignore_index=True)
combined_significant = combined_enrichment.loc[combined_enrichment["significantly_overrepresented"]].copy()
combined_contributors = pd.concat([result["contributors"] for result in comparison_results.values()], ignore_index=True)

combined_overview.to_csv(OUTPUT_DIR / "combined_pipeline_overview.csv", index=False)
combined_coverage.to_csv(OUTPUT_DIR / "combined_unique_marker_coverage.csv", index=False)
combined_enrichment.to_csv(OUTPUT_DIR / "combined_unique_marker_enrichment_all_tests.csv", index=False)
combined_significant.to_csv(OUTPUT_DIR / "combined_unique_marker_enrichment_significant.csv", index=False)
combined_contributors.to_csv(OUTPUT_DIR / "combined_genes_contributing_to_significant_enrichment.csv", index=False)

display(combined_overview)
display(combined_significant)

## Plot significant unique-marker enrichments

In [ ]:
for comparison in DESEQ_TABLES:
    plot_data = combined_significant.loc[combined_significant["comparison"].eq(comparison)].copy()
    plot_data = plot_data.sort_values(["direction", "fdr"], ascending=[True, False])

    if plot_data.empty:
        print(f"{comparison}: no unique cell-type marker sets are significantly enriched at FDR < {FDR_CUTOFF}.")
        continue

    plot_data["label"] = plot_data["direction"] + " | " + plot_data["celltype"]
    plot_data["minus_log10_fdr"] = -np.log10(plot_data["fdr"].clip(lower=np.finfo(float).tiny))
    colors = plot_data["direction"].map({"upregulated": "#C44E52", "downregulated": "#4C72B0"})

    fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(plot_data))))
    ax.barh(plot_data["label"], plot_data["minus_log10_fdr"], color=colors)
    ax.axvline(-np.log10(FDR_CUTOFF), color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("-log10(FDR)")
    ax.set_title(f"Unique cell-type marker enrichment: {comparison}")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{comparison}_significant_unique_marker_enrichment.png", dpi=300, bbox_inches="tight")
    plt.show()

display(combined_contributors.head(30))

In [ ]:
comparison = "EcoKD1_Eco1KD_B8_vs_control_B8"
plot_data = combined_significant.loc[combined_significant["comparison"].eq(comparison)].copy()
plot_data = plot_data.sort_values(["direction", "fdr"], ascending=[True, False])

if plot_data.empty:
    print(f"{comparison}: no unique cell-type marker sets are significantly enriched at FDR < {FDR_CUTOFF}.")

plot_data["label"] = plot_data["direction"] + " | " + plot_data["celltype"]
plot_data["minus_log10_fdr"] = -np.log10(plot_data["fdr"].clip(lower=np.finfo(float).tiny))
colors = plot_data["direction"].map({"upregulated": "#C44E52", "downregulated": "#4C72B0"})

fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(plot_data))))
ax.barh(plot_data["label"], plot_data["minus_log10_fdr"], color=colors)
ax.axvline(-np.log10(FDR_CUTOFF), color="black", linestyle="--", linewidth=1)
ax.set_xlabel("-log10(FDR)")
ax.set_title(f"Unique cell-type marker enrichment: {comparison}")
fig.tight_layout()
#fig.savefig(OUTPUT_DIR / f"{comparison}_significant_unique_marker_enrichment.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_data

In [ ]:
celltype_order = ["C_nematoblast","C_nematocytes","C_tentacle","C_EC_foot",
                  "C_EN_foot","C_gland_cell","I_ZymoGl","Ec_BasalDisk","I_GranGl","Ec_Peduncle","I_En2N"].reverse()

In [ ]:
fig_height = 9
fig_width = 9
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

celltype_order = ["C_nematoblast","C_nematocytes","C_tentacle","C_EC_foot",
                  "C_EN_foot","C_gland_cell"][::-1] # "I_ZymoGl","Ec_BasalDisk","I_GranGl","Ec_Peduncle","I_En2N"
experiment_labels = ["downregulated", "upregulated"]

x_lookup = {name: i + 1 for i, name in enumerate(experiment_labels)}
y_lookup = {name: i + 1 for i, name in enumerate(celltype_order)}
x = plot_data["direction"].map(x_lookup)
y = plot_data["celltype"].map(y_lookup)


# Scale bubble areas while keeping small counts visible.
marker_counts = plot_data["selected_markers"].astype(float)
size_scale = 15
sizes = np.maximum(marker_counts, 1) * size_scale

scatter = ax.scatter(
    x,
    y,
    s=sizes,
    c=plot_data["minus_log10_fdr"],
    cmap="YlGnBu",
    alpha=0.85,
    edgecolors="black",
    linewidths=0.5,
)

# One empty tick before and after the actual data on both axes
x_ticks = range(len(experiment_labels) + 2)
x_labels = [""] + experiment_labels + [""]

y_ticks = range(len(celltype_order) + 2)
y_labels = [""] + celltype_order + [""]

ax.set_xticks(x_ticks)
ax.set_xticklabels(["","Eco KD down","Eco KD up",""], rotation=45, ha="right", fontsize=18)

ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=18)

ax.set_xlim(0.5, len(experiment_labels) + 0.5)
ax.set_ylim(0.5, len(celltype_order) + 0.5)

#ax.set_xlabel("Experiment")
#ax.set_ylabel("Enriched cell type")
#ax.set_title(f"Cell-type enrichment among {direction} genes")
ax.grid(True, axis="both", linestyle=":", linewidth=0.6, alpha=0.5)
ax.set_axisbelow(True)

# Bubble-size legend based on representative observed counts.
observed = np.sort(plot_data["selected_markers"].unique())
#if len(observed) <= 4:
#    legend_counts = observed
#else:
#    idx = np.unique(np.linspace(0, len(observed) - 1, 4).round().astype(int))
#    legend_counts = observed[idx]


legend_counts = np.array([10,20,50,100])
handles = [
    ax.scatter([], [], s=max(float(n), 1) * size_scale, facecolors="none",
               edgecolors="black", linewidths=0.8, vmin=0)
    for n in legend_counts
]

# Reserve space on the right side of the figure
fig.subplots_adjust(right=0.72)

# Position of the main plotting axes in figure coordinates
pos = ax.get_position()

# ------------------------------------------------------------
# Colorbar: lower half of main y-axis
# ------------------------------------------------------------
cax = fig.add_axes([
    pos.x1 + 0.025,
    pos.y0,
    0.025,
    pos.height * 0.48
])

cbar = fig.colorbar(scatter, cax=cax)

cbar.set_label(
    "Enrichment strength, -log10(FDR)",
    labelpad=10,
    fontsize=18
)

cbar.ax.tick_params(axis="y", labelsize=18)

# ------------------------------------------------------------
# Bubble-size legend: upper half of main y-axis
# ------------------------------------------------------------
legend_ax = fig.add_axes([
    pos.x1 + 0.02,          # x
    pos.y0 + pos.height * 0.52,
    0.20,                   # width
    pos.height * 0.48       # height
])

legend_ax.set_xlim(0, 1)
legend_ax.set_ylim(0, 1)

# Completely transparent auxiliary axes
legend_ax.set_facecolor("none")
legend_ax.axis("off")

# Make sure legend is drawn above everything else
legend_ax.set_zorder(10)


# Bubble positions
legend_y = np.linspace(
    0.12,
    0.78,
    len(legend_counts)
)[::-1]

for ypos, n in zip(legend_y, legend_counts):

    legend_ax.scatter(
        0.20,
        ypos,
        s=max(float(n), 1) * size_scale,
        facecolors="none",
        edgecolors="black",
        linewidths=0.8,
        clip_on=False,          # important!
        zorder=20,
    )

    legend_ax.text(
        0.43,
        ypos,
        str(int(n)),
        va="center",
        ha="left",
        clip_on=False,
        zorder=20,
        fontsize=18
    )

legend_ax.text(
    0.0,
    0.98,
    "DE genes in cell type",
    ha="left",
    va="top",
    fontweight="bold",
    clip_on=False,
    zorder=20,
)

plt.savefig("results/unique_marker_celltype_enrichment_deseq2_comparisons/ecoKD_bubble_plot.svg")

In [ ]:
celltype_order = ["C_nematoblast","C_nematocytes","C_tentacle","C_EC_foot",
                  "C_EN_foot","C_gland_cell","I_ZymoGl","EC_BasalDisk","I_GranGl","Ec_Peduncle","I_En2N"]

In [ ]:
-np.log(0.05)

In [ ]:
filtered_eco = combined_contributors[combined_contributors["comparison"] == "EcoKD1_Eco1KD_B8_vs_control_B8"]

In [ ]:
filtered_eco[filtered_eco["celltype"] == "Ec_Peduncle"]